In [6]:
from pathlib import Path
from time import perf_counter

import torch
import torch.nn.functional as F

from torchcodec.decoders import VideoDecoder
from torchvision.models import resnet18, ResNet18_Weights

In [7]:
from ml_systems.environment import print_environment

print_environment()

PyTorch: 2.8.0+cu128
CUDA runtime: 12.8
CUDA available: True
GPU: NVIDIA RTX 2000 Ada Generation


In [8]:
VIDEO_PATH = Path("../data/clips/sample.mp4")

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA RTX 2000 Ada Generation


In [9]:
decoder = VideoDecoder(str(VIDEO_PATH))

print(decoder.metadata)

VideoStreamMetadata:
  duration_seconds_from_header: 25.441667
  begin_stream_seconds_from_header: 0.0
  bit_rate: 2499384.0
  codec: h264
  stream_index: 0
  begin_stream_seconds_from_content: 0.0
  end_stream_seconds_from_content: 25.442083
  width: 1280
  height: 720
  num_frames_from_header: 610
  num_frames_from_content: 610
  average_fps_from_header: 23.976024
  pixel_aspect_ratio: 1
  duration_seconds: 25.442083
  begin_stream_seconds: 0.0
  end_stream_seconds: 25.442083
  num_frames: 610
  average_fps: 23.976024290149514



In [10]:
metadata = decoder.metadata

print("Number of frames:", metadata.num_frames)
print("FPS:", metadata.average_fps)
print("Width:", metadata.width)
print("Height:", metadata.height)
print("Duration:", metadata.duration_seconds)
print("Codec:", metadata.codec)

Number of frames: 610
FPS: 23.976024290149514
Width: 1280
Height: 720
Duration: 25.442083
Codec: h264


In [11]:
frame = decoder[0]

print("Shape:", frame.shape)
print("dtype:", frame.dtype)
print("device:", frame.device)
print("min:", frame.min())
print("max:", frame.max())

Shape: torch.Size([3, 720, 1280])
dtype: torch.uint8
device: cpu
min: tensor(36, dtype=torch.uint8)
max: tensor(255, dtype=torch.uint8)


In [12]:
print("Number of values:", frame.numel())
print("Bytes per value:", frame.element_size())

frame_mb = frame.numel() * frame.element_size() / 1024**2

print(f"Frame memory: {frame_mb:.2f} MB")

Number of values: 2764800
Bytes per value: 1
Frame memory: 2.64 MB


In [13]:
num_frames = decoder.metadata.num_frames
num_samples = 8

indices = torch.linspace(
    0,
    num_frames - 1,
    steps=num_samples
).long()

print(indices)

tensor([  0,  87, 174, 261, 348, 435, 522, 609])


In [14]:
frames = []

for idx in indices:
    frame = decoder[int(idx)]
    frames.append(frame)

frames = torch.stack(frames)

print(frames.shape)

torch.Size([8, 3, 720, 1280])


In [15]:
frames_mb = (
    frames.numel()
    * frames.element_size()
    / 1024**2
)

print(f"{frames_mb:.2f} MiB") 

21.09 MiB


In [16]:
frames = frames.float() / 255.0

print(frames.dtype)
print(frames.min())
print(frames.max())

torch.float32
tensor(0.0824)
tensor(1.)


In [17]:
frames = F.interpolate(
    frames,
    size=(224, 224),
    mode="bilinear",
    align_corners=False,
)

print(frames.shape)

torch.Size([8, 3, 224, 224])


In [18]:
mean = torch.tensor(
    [0.485, 0.456, 0.406]
).view(1, 3, 1, 1)

std = torch.tensor(
    [0.229, 0.224, 0.225]
).view(1, 3, 1, 1)

frames = (frames - mean) / std

In [19]:
print("Shape:", frames.shape)
print("dtype:", frames.dtype)
print("device:", frames.device)
print("min:", frames.min())
print("max:", frames.max())

Shape: torch.Size([8, 3, 224, 224])
dtype: torch.float32
device: cpu
min: tensor(-1.5824)
max: tensor(2.6400)


In [20]:
frames_mb = (
    frames.numel()
    * frames.element_size()
    / 1024**2
)

print(f"{frames_mb:.2f} MiB")

4.59 MiB


In [21]:
device = torch.device("cuda")

weights = ResNet18_Weights.DEFAULT

model = resnet18(weights=weights)
model.eval()
model = model.to(device)

print("GPU:", torch.cuda.get_device_name(0))
print("Model device:", next(model.parameters()).device)

GPU: NVIDIA RTX 2000 Ada Generation
Model device: cuda:0


CPU RAM                         GPU VRAM

frames                          ResNet18
[8,3,224,224]                   parameters

In [22]:
# CPU → GPU transfer

print("Before:", frames.device)

frames_gpu = frames.to(device)

print("After:", frames_gpu.device)

Before: cpu
After: cuda:0


In [ ]:
# GPU inference

with torch.inference_mode():
    output = model(frames_gpu)

print("Output shape:", output.shape)
print("Output device:", output.device)

Output shape: torch.Size([8, 1000])
Output device: cuda:0


1000 ImageNet scores per frame

In [24]:
probabilities = output.softmax(dim=1)

top_probs, top_indices = probabilities.max(dim=1)

categories = weights.meta["categories"]

for i, (idx, prob) in enumerate(zip(top_indices, top_probs)):
    label = categories[idx.item()]

    print(
        f"Frame {i}: "
        f"{label:<25} "
        f"{prob.item():.3f}"
    )

Frame 0: carousel                  0.096
Frame 1: acoustic guitar           0.580
Frame 2: plunger                   0.080
Frame 3: stage                     0.538
Frame 4: stage                     0.132
Frame 5: stage                     0.106
Frame 6: pajama                    0.303
Frame 7: candle                    0.106


In [25]:
# Benchmark CPU → GPU transfer

torch.cuda.synchronize()

start = perf_counter()

frames_gpu = frames.to(device)

torch.cuda.synchronize()

transfer_time = perf_counter() - start

print(f"CPU → GPU transfer: {transfer_time * 1000:.3f} ms")

CPU → GPU transfer: 10.196 ms


In [27]:
# warm up GPU before benchmarking inference

for _ in range(5):
    with torch.inference_mode():
        _ = model(frames_gpu)

torch.cuda.synchronize()

# Benchmark GPU inference

inference_times = []

for _ in range(20):

    torch.cuda.synchronize()

    start = perf_counter()

    with torch.inference_mode():
        output = model(frames_gpu)

    torch.cuda.synchronize()

    elapsed = perf_counter() - start

    inference_times.append(elapsed * 1000)

In [28]:
import statistics

print(f"Mean:   {statistics.mean(inference_times):.3f} ms")
print(f"Median: {statistics.median(inference_times):.3f} ms")
print(f"Min:    {min(inference_times):.3f} ms")
print(f"Max:    {max(inference_times):.3f} ms")

Mean:   4.062 ms
Median: 4.058 ms
Min:    4.040 ms
Max:    4.122 ms


In [ ]:
# GPU memory

# clear temp cache and reset peak counter 
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()